In [ ]:
from __future__ import division
from IPython.display import display
from datetime import datetime
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from my_code.utils import show_videos
from my_code.plotting import plot_channel_stats, plot_compressed_channel_stats, plot_sentiment_series
%matplotlib inline
channels = pd.read_csv('channels.csv')
topics = pd.read_csv('topics.csv')

from my_code.youtube_api import download_channels_videos
download_channels_videos(channels)

from my_code.youtube_api import merge_channel_videos
merge_channel_videos(channels)
from my_code.utils import create_topic_columns
videos = pd.read_csv('videos-MERGED.csv')
create_topic_columns(videos, topics)
videos.to_csv('videos.csv', index=False, encoding='utf-8')
from my_code.language_api import download_sentiments
download_sentiments(videos)

In [ ]:
global all_videos
all_videos = pd.read_csv('videos.csv', parse_dates=['published_at'])
all_videos.head(3)
num_relevant = all_videos.relevant.sum()
num_total = all_videos.shape[0]
print ('Number of relevant videos: %s' % num_relevant)
print ('Total number of videos: %s' % num_total)
print ('Percentage of relevant videos: %0.2f%%' % (100*num_relevant/num_total))

In [ ]:
channel_stats = pd.DataFrame({
    'relevant': all_videos.groupby('channel').relevant.sum().astype(int),
    'total': all_videos.groupby('channel').size()
})
channel_stats['percentage_relevant'] = (100*channel_stats.relevant/channel_stats.total).round(2)
channel_stats.sort_values('percentage_relevant', ascending=False)

In [ ]:
absolutes = all_videos.groupby('channel')[topics.slug].sum().astype(int)
display(absolutes)

In [ ]:
totals = all_videos.groupby('channel').size()
relatives = 100 * absolutes.divide(totals, axis=0)
display(relatives)

In [ ]:
plot_channel_stats(relatives, topics, channels, title='Relative topic coverage\n(% of total # of each channel\'s videos)')

In [ ]:
sentiments = pd.read_csv('sentiments.csv')
sentiments[['youtube_id', 'sentiment_score']].head()

In [ ]:
videos = all_videos[all_videos.relevant].merge(sentiments, on='youtube_id')
videos.sort_values('sentiment_score')[['channel', 'title', 'sentiment_score', 'youtube_id']].head(4)

In [ ]:
sns.displot(videos.sentiment_score, kde=True)
plt.title('Sentiment Scores Distribution')
plt.gca().get_yaxis().set_visible(False)
plt.xlim(-1, 1)
plt.show()


In [ ]:
scores = pd.DataFrame(
    index=channels.sort_values('title').title,
    columns=topics.slug
)
for channel, group in videos.groupby('channel'):
    for topic in topics.slug:
        topic_data = group[group[topic]].sentiment_score
        if not topic_data.empty:
            scores.loc[channel, topic] = np.nanmean(topic_data)
        else:
            scores.loc[channel, topic] = np.nan 
scores = scores.rename_axis('Channel', axis=0)
scores = scores.rename_axis('Topic', axis=1)
scores = scores.fillna(0)
display(scores)
plot_channel_stats(scores,topics,channels,fig_height=10,y_center=True,title='Average sentiment by topic')


In [ ]:
scores = scores.apply(pd.to_numeric, errors='coerce') 
scores = scores.fillna(0)                         
plot_compressed_channel_stats(scores, y_center=True, title='Average sentiment by topic')